# [Day 2 종합실습] NYC Yellow Taxi — 데이터 준비 + EDA

NYC Yellow Taxi 2026-05 trip 데이터를 Pandas/Polars 양쪽으로 로딩해 결과를 비교하고,  
결측치·중복 행을 정리한 뒤 기본 EDA 및 시각화를 수행한다.

---
**실행 순서**: 위에서 아래로 셀을 순서대로 실행 (`Run All` 또는 `Shift+Enter`)  
**데이터**: 원본 URL에서 자동 다운로드 → `data/` 폴더에 캐시 저장 (이미 있으면 스킵)  
**결과물**: `output/` 폴더에 시각화 PNG 저장

| Part | 내용 |
|------|------|
| 1 | 환경 설정 및 함수 정의 |
| 2 | 데이터 로딩 — Pandas vs Polars 비교 |
| 3 | 데이터 정제 (py 원본 재현) |
| 4 | Zone 매핑 + py 기본 EDA 재현 |
| 5 | 정제 후 데이터 상태 점검 |
| 6 | 데이터 미리보기 (head / tail / sample) |
| 7 | 컬럼 타입 자동 분류 |
| 8 | 기술통계 — describe(include=all) |
| 9 | 범주형 변수 종합 분석 |
| 10 | 수치형 왜도/첨도 분석 |
| 11 | 수치형 분포 — 히스토그램 + KDE |
| 12 | 수치형 이상치 — 박스플롯 |
| 13 | 수치형 vs 수치형 — 상관관계 히트맵 |
| 14 | 수치형 vs 수치형 — 산점도 |
| 15 | 수치형 vs 범주형 — 그룹별 박스플롯 |
| 16 | 핵심 인사이트 시각화 |
| 17 | Borough 위치 분석 |
| 18 | 관찰 정리 + 팀원 인계 메모 |

---
## Part 1. 환경 설정 및 함수 정의

In [ ]:
import subprocess
subprocess.run(["pip", "install", "-q", "kaleido"], check=True)
print("kaleido 설치 완료")

In [ ]:
from __future__ import annotations
import math, platform, sys, timeit, urllib.request
from pathlib import Path
from typing import Callable
from urllib.error import URLError

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import polars as pl
import seaborn as sns
from plotly.subplots import make_subplots

import matplotlib.font_manager as fm

if platform.system() == "Darwin":
    # 폰트 캐시에서 AppleGothic 직접 경로 찾기
    font_list = [f.name for f in fm.fontManager.ttflist]
    if "AppleGothic" in font_list:
        plt.rcParams["font.family"] = "AppleGothic"
        PLOTLY_FONT = "AppleGothic"
    else:
        # AppleGothic 없으면 나눔고딕 시도
        plt.rcParams["font.family"] = "NanumGothic"
        PLOTLY_FONT = "NanumGothic"
else:
    plt.rcParams["font.family"] = "DejaVu Sans"
    PLOTLY_FONT = "DejaVu Sans"

plt.rcParams["axes.unicode_minus"] = False

# 폰트 캐시 강제 갱신
fm._load_fontmanager(try_read_cache=False)

print(f"설정된 폰트: {plt.rcParams['font.family']}")
print(f"사용 가능한 한글 폰트: {[f.name for f in fm.fontManager.ttflist if 'Gothic' in f.name or 'Nanum' in f.name]}")
# ── 경로 상수 ──
BASE_DIR         = Path(".").resolve()
DATA_DIR         = BASE_DIR / "data"
OUTPUT_DIR       = BASE_DIR / "output"
RAW_PATH         = DATA_DIR / "yellow_tripdata_2026-05.parquet"
ZONE_LOOKUP_PATH = DATA_DIR / "taxi_zone_lookup.csv"
CLEANED_PATH     = DATA_DIR / "nyc_taxi_cleaned.parquet"

DATA_URL        = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-05.parquet"
ZONE_LOOKUP_URL = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

NULLABLE_COLUMNS = ["passenger_count", "RatecodeID", "store_and_fwd_flag",
                    "congestion_surcharge", "Airport_fee"]
BENCHMARK_NUMBER = 3

# ── 범주형 라벨 매핑 ──
LABEL_MAP = {
    "payment_type"      : {1:"신용카드", 2:"현금", 3:"무료", 4:"분쟁", 5:"미상", 6:"취소"},
    "VendorID"          : {1:"CMT", 2:"VeriFone"},
    "RatecodeID"        : {1:"표준", 2:"JFK고정", 3:"Newark", 4:"Nassau",
                           5:"협상", 6:"그룹", 99:"기타"},
    "store_and_fwd_flag": {"Y":"오프라인→전송", "N":"실시간전송"},
}

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
print(f"BASE_DIR: {BASE_DIR}")
print(f"OS: {platform.system()} | 폰트: {PLOTLY_FONT}")

In [ ]:
def print_title(title: str) -> None:
    print(f"\n{'='*70}\n{title}\n{'='*70}")

def benchmark(name: str, fn: Callable) -> dict:
    secs = timeit.timeit(fn, number=BENCHMARK_NUMBER)
    return {"tool": name, "seconds": secs / BENCHMARK_NUMBER}

def ensure_raw_data() -> Path:
    if not RAW_PATH.exists():
        print(f"다운로드 중: {DATA_URL}")
        try:
            urllib.request.urlretrieve(DATA_URL, RAW_PATH)
            print("완료")
        except (URLError, OSError, TimeoutError) as e:
            raise RuntimeError(f"다운로드 실패: {e}") from e
    else:
        print(f"캐시 사용: {RAW_PATH}")
    return RAW_PATH

def ensure_zone_lookup() -> Path:
    if not ZONE_LOOKUP_PATH.exists():
        print(f"다운로드 중: {ZONE_LOOKUP_URL}")
        try:
            urllib.request.urlretrieve(ZONE_LOOKUP_URL, ZONE_LOOKUP_PATH)
            print("완료")
        except (URLError, OSError, TimeoutError) as e:
            raise RuntimeError(f"다운로드 실패: {e}") from e
    else:
        print(f"캐시 사용: {ZONE_LOOKUP_PATH}")
    return ZONE_LOOKUP_PATH

def load_with_pandas(path: Path) -> pd.DataFrame:
    return pd.read_parquet(path)

def load_with_polars(path: Path) -> pl.DataFrame:
    return pl.read_parquet(path)

def save_fig(fig_obj, filename: str) -> None:
    out = OUTPUT_DIR / filename
    if isinstance(fig_obj, plt.Figure):
        fig_obj.savefig(out, dpi=150, bbox_inches="tight")
    else:
        fig_obj.write_image(str(out), scale=2)
    print(f"저장: {out}")

print("함수 정의 완료")

---
## Part 2. 데이터 로딩 — Pandas vs Polars 비교

In [ ]:
raw_path = ensure_raw_data()
pdf  = load_with_pandas(raw_path)
pldf = load_with_polars(raw_path)

print_title("1. Pandas vs Polars 로딩 결과 비교")
print(f"Pandas shape : {pdf.shape}")
print(f"Polars shape : {pldf.shape}")
print(f"행 수 일치   : {pdf.shape[0] == pldf.shape[0]}")

pandas_na = pdf[NULLABLE_COLUMNS].isna().sum()
polars_na = pldf.select(NULLABLE_COLUMNS).null_count()
print("\n[컬럼별 결측치 수 비교]")
for col in NULLABLE_COLUMNS:
    p, pl_ = int(pandas_na[col]), int(polars_na[col][0])
    print(f"  {col:22s} Pandas={p:8,d}  Polars={pl_:8,d}  일치={p==pl_}")

print(f"\n[로딩 속도 비교 ({BENCHMARK_NUMBER}회 평균)]")
for r in sorted([
    benchmark("Pandas", lambda: load_with_pandas(raw_path)),
    benchmark("Polars", lambda: load_with_polars(raw_path)),
], key=lambda x: x["seconds"]):
    print(f"  {r['tool']:8s} 평균 {r['seconds']:.3f} s")

---
## Part 3. 데이터 정제 (py 원본 재현)

In [ ]:
def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    print_title("2. 결측치·중복 처리")
    dup = int(df.duplicated().sum())
    cleaned = df.drop_duplicates().copy()
    print(f"중복 행 제거: {dup:,}건 ({len(df):,} → {len(cleaned):,})")

    med = cleaned["passenger_count"].median()
    mis = int(cleaned["passenger_count"].isna().sum())
    cleaned["passenger_count"] = cleaned["passenger_count"].fillna(med)
    print(f"passenger_count: 결측 {mis:,}건 → 중앙값 {med}로 대체")

    for col in ["RatecodeID", "store_and_fwd_flag"]:
        mode_val = cleaned[col].mode()[0]
        mis = int(cleaned[col].isna().sum())
        cleaned[col] = cleaned[col].fillna(mode_val)
        print(f"{col}: 결측 {mis:,}건 → 최빈값 '{mode_val}'로 대체")

    for col in ["congestion_surcharge", "Airport_fee"]:
        mis = int(cleaned[col].isna().sum())
        cleaned[col] = cleaned[col].fillna(0)
        print(f"{col}: 결측 {mis:,}건 → 0으로 대체 (요금 미부과)")

    return cleaned

cleaned = clean_data(pdf)

---
## Part 4. Zone 매핑 + py 기본 EDA 재현

In [ ]:
def run_basic_eda(df: pd.DataFrame) -> None:
    print_title("3. 기본 EDA (py 원본)")
    print(f"shape: {df.shape}")
    print("\n[수치형 기술통계]")
    print(df[["trip_distance","fare_amount","tip_amount","total_amount"]].describe())
    print("\n[payment_type 분포]")
    print(df["payment_type"].value_counts().sort_index())
    dur = (df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]).dt.total_seconds()/60
    print(f"\n[소요시간] 평균 {dur.mean():.1f}분, 중앙값 {dur.median():.1f}분")
    print(f"소요시간 0분 이하: {(dur<=0).sum():,}건")
    q1, q3 = df["total_amount"].quantile(0.25), df["total_amount"].quantile(0.75)
    iqr = q3-q1; lo, hi = q1-1.5*iqr, q3+1.5*iqr
    out = ~df["total_amount"].between(lo,hi)
    print(f"\n[total_amount IQR 이상치] [{lo:.2f}, {hi:.2f}]")
    print(f"이상치 {out.sum():,}건 ({out.mean()*100:.1f}%) | 음수 {(df['total_amount']<0).sum():,}건")

def map_location_names(df: pd.DataFrame, zone_lookup: pd.DataFrame) -> pd.DataFrame:
    lookup = zone_lookup.set_index("LocationID")[["Borough","Zone"]]
    result = df.join(lookup.add_prefix("PU_"), on="PULocationID")
    return result.join(lookup.add_prefix("DO_"), on="DOLocationID")

def show_location_map(df: pd.DataFrame) -> None:
    print_title("4. 위치(Zone) 매핑")
    bs = (df.groupby("PU_Borough",observed=True)
          .agg(trips=("total_amount","count"),avg_total=("total_amount","mean"))
          .sort_values("trips",ascending=False))
    print(bs)
    top = df["PU_Zone"].value_counts().head(10)
    print("\n[픽업 Zone 상위 10]\n", top)
    fig, ax = plt.subplots(figsize=(10,6))
    top.sort_values().plot(kind="barh", ax=ax, color="steelblue")
    ax.set_title("픽업 트립 수 상위 10 Zone")
    ax.set_xlabel("트립 수")
    plt.tight_layout()
    save_fig(fig, "00_top_zones_py_original.png")
    plt.show(); plt.close(fig)

run_basic_eda(cleaned)
zone_lookup = pd.read_csv(ensure_zone_lookup())
mapped = map_location_names(cleaned, zone_lookup)
show_location_map(mapped)
cleaned.to_parquet(CLEANED_PATH, index=False)
print(f"\n정제 데이터 저장: {CLEANED_PATH}")

---
## Part 5. 정제 후 데이터 상태 점검

In [ ]:
print_title("5. 정제 후 데이터 상태 점검")
print(f"shape: {cleaned.shape}  (행: {cleaned.shape[0]:,} / 열: {cleaned.shape[1]})")
print(f"원본 대비: {len(pdf):,} → {len(cleaned):,} ({len(pdf)-len(cleaned):,}행 제거)")
print("\n[컬럼 정보 요약]")
col_info = pd.DataFrame({
    "dtype"    : cleaned.dtypes,
    "결측치 수" : cleaned.isnull().sum(),
    "결측 비율" : (cleaned.isnull().sum()/len(cleaned)*100).round(2).astype(str)+"%",
    "고유값 수" : cleaned.nunique(),
    "샘플값"   : cleaned.iloc[0],
})
display(col_info)
print(f"\n중복 행: {cleaned.duplicated().sum():,}건  (0이어야 정상)")
remaining_na = cleaned.isnull().sum()
remaining_na = remaining_na[remaining_na > 0]
if remaining_na.empty:
    print("결측치: 모든 컬럼 결측치 없음 ✓")
else:
    print(f"결측치 남은 컬럼:\n{remaining_na}")

---
## Part 6. 데이터 미리보기 (head / tail / sample)
> 강의자료 EDA Checklist p.17~18

In [ ]:
print_title("6. 데이터 미리보기")
print("\n[head(5)] — 처음 5행")
display(cleaned.head(5))
print("\n[tail(5)] — 마지막 5행")
display(cleaned.tail(5))
print("\n[sample(5, random_state=42)] — 무작위 5행")
display(cleaned.sample(5, random_state=42))

---
## Part 7. 컬럼 타입 자동 분류

In [ ]:
ID_LIKE_COLS  = {"VendorID","PULocationID","DOLocationID",
                 "payment_type","RatecodeID","passenger_count"}
DATETIME_COLS = [c for c in cleaned.columns
                 if pd.api.types.is_datetime64_any_dtype(cleaned[c])]
LOCATION_COLS = ["PULocationID","DOLocationID"]

numeric_cols = [
    c for c in cleaned.select_dtypes(include=["number"]).columns
    if c not in ID_LIKE_COLS
]
categorical_cols = (
    list(cleaned.select_dtypes(include=["object","category"]).columns)
    + [c for c in ID_LIKE_COLS if c in cleaned.columns and c not in LOCATION_COLS]
)
categorical_cols = list(dict.fromkeys(categorical_cols))

print_title("7. 컬럼 타입 분류 결과")
print(f"\n■ 수치형   ({len(numeric_cols)}개): {numeric_cols}")
print(f"\n■ 범주형   ({len(categorical_cols)}개): {categorical_cols}")
print(f"\n■ 날짜형   ({len(DATETIME_COLS)}개): {DATETIME_COLS}")
print(f"\n■ 위치코드 ({len(LOCATION_COLS)}개): {LOCATION_COLS}")
print(f"\n전체 컬럼 수: {cleaned.shape[1]}개")

---
## Part 8. 기술통계 — describe(include='all')
> 강의자료 p.19 — 수치형 + 범주형 전체 요약

In [ ]:
print_title("8-A. 수치형 기술통계")
display(
    cleaned[numeric_cols].describe().T
    .style.format("{:.3f}")
    .background_gradient(cmap="Blues", axis=0)
)

In [ ]:
print_title("8-B. 전체 기술통계 describe(include='all')")
display(cleaned.describe(include="all"))

---
## Part 9. 범주형 변수 종합 분석
> 강의자료 p.20 체크리스트: unique / nunique / value_counts / value_counts(normalize=True)
> 강의자료 p.27 시각화: Bar Chart

In [ ]:
print_title("9-A. 범주형 변수 요약표")
rows = []
for col in categorical_cols:
    vc = cleaned[col].value_counts()
    rows.append({
        "컬럼명"     : col,
        "dtype"      : str(cleaned[col].dtype),
        "고유값 수"   : cleaned[col].nunique(),
        "결측치"     : cleaned[col].isnull().sum(),
        "최빈값"     : vc.index[0],
        "최빈값 비율" : f"{vc.iloc[0]/len(cleaned)*100:.1f}%",
        "고유값 목록" : str(sorted(cleaned[col].dropna().unique().tolist())),
    })
display(pd.DataFrame(rows).set_index("컬럼명"))

In [ ]:
print_title("9-B. 컬럼별 unique / value_counts / 상대도수")
for col in categorical_cols:
    vc     = cleaned[col].value_counts()
    vc_pct = cleaned[col].value_counts(normalize=True).mul(100).round(2)
    detail = pd.DataFrame({"도수": vc, "비율(%)": vc_pct})
    if col in LABEL_MAP:
        detail.index = detail.index.map(
            lambda x: f"{x} ({LABEL_MAP[col].get(x,'?')})"
        )
    print(f"\n── {col}")
    print(f"   unique 값: {cleaned[col].unique().tolist()}")
    display(detail)

In [ ]:
print_title("9-C. 범주형 분포 — Bar Chart (강의자료 p.27 권장)")
n_c = 3; n_r = math.ceil(len(categorical_cols)/n_c)
fig, axes = plt.subplots(n_r, n_c, figsize=(18, n_r*5))
axes = axes.flatten()

for i, col in enumerate(categorical_cols):
    ax = axes[i]
    vc = cleaned[col].value_counts().reset_index()
    vc.columns = [col, "count"]
    vc["pct"] = (vc["count"]/len(cleaned)*100).round(1)
    vc["label"] = vc[col].map(LABEL_MAP.get(col,{})).fillna(vc[col].astype(str)) if col in LABEL_MAP else vc[col].astype(str)

    sns.barplot(data=vc, x="label", y="count", ax=ax,
                palette="Blues_d", order=vc["label"])
    for j, row in vc.iterrows():
        ax.text(j, row["count"]*1.01, f"{row['pct']}%",
                ha="center", va="bottom", fontsize=9)
    ax.set_title(f"{col}\n(고유값 {cleaned[col].nunique()}개 | 최빈={vc.iloc[0]['label']})", fontsize=10)
    ax.set_xlabel(""); ax.set_ylabel("트립 수")
    ax.tick_params(axis="x", rotation=20)

for j in range(i+1, len(axes)): fig.delaxes(axes[j])
fig.suptitle("범주형 컬럼 분포 — Bar Chart", fontsize=14, y=1.01)
plt.tight_layout()
save_fig(fig, "09_categorical_bar.png")
plt.show(); plt.close(fig)

print("\n[강의자료 Think to look for — 범주형]")
print("  ✓ High Cardinality 변수? (nunique ≈ 행 수 → Encoding 불가)")
print("  ✓ 특정 값에 극단 치우침? → 클래스 불균형 주의")
print("  ✓ 의미 없는 범주(Unknown 등)가 많은 비중?")
for col in categorical_cols:
    top_pct = cleaned[col].value_counts(normalize=True).iloc[0]*100
    if top_pct > 80:
        print(f"  ⚠ {col}: 최빈값 {top_pct:.1f}% → 클래스 불균형 확인 필요")

---
## Part 10. 수치형 — 왜도(Skewness) / 첨도(Kurtosis)
> 강의자료 p.15 Descriptive Statistics — 분포 형태 파악
> 왜도 |v|>1 → 로그변환 권장 (강의자료 p.57)

In [ ]:
print_title("10. 수치형 왜도 / 첨도 분석")
skew = cleaned[numeric_cols].skew()
kurt = cleaned[numeric_cols].kurtosis()  # 초과첨도 (정규=0)

def skew_label(v):
    if abs(v) <= 0.5:   return "대칭 (변환 불필요)"
    elif abs(v) <= 1.0: return "약한 비대칭 (변환 고려)"
    else:               return "강한 비대칭 → 로그변환 권장"

def kurt_label(v):
    if v > 1:    return "뾰족 (이상치 많을 가능성)"
    elif v < -1: return "납작 (꼬리 얇음)"
    else:        return "정상 범위"

display(pd.DataFrame({
    "왜도(skewness)" : skew.round(3),
    "왜도 해석"      : skew.map(skew_label),
    "첨도(kurtosis)": kurt.round(3),
    "첨도 해석"      : kurt.map(kurt_label),
}))

log_cands = skew[skew.abs() > 1].index.tolist()
print(f"\n로그변환 권장 컬럼 ({len(log_cands)}개): {log_cands}")
print("→ np.log1p() 사용 (0 포함 데이터 대응)")

---
## Part 11. 수치형 분포 — 히스토그램 + KDE
> 강의자료 p.26 Quantitative Variable 살펴보기

In [ ]:
n_c = 3; n_r = math.ceil(len(numeric_cols)/n_c)
fig, axes = plt.subplots(n_r, n_c, figsize=(18, n_r*4))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    ax = axes[i]
    data = cleaned[col].dropna()
    p1, p99 = data.quantile(0.01), data.quantile(0.99)
    data_c = data[(data>=p1)&(data<=p99)]
    skew_v = data.skew()
    sns.histplot(data_c, kde=True, ax=ax, color="steelblue", alpha=0.7)
    ax.set_title(f"{col}\n왜도={skew_v:.2f} | 1~99th pct", fontsize=10)
    ax.set_xlabel(col); ax.set_ylabel("빈도")

for j in range(i+1, len(axes)): fig.delaxes(axes[j])
fig.suptitle("수치형 컬럼 분포 — 히스토그램 + KDE", fontsize=14, y=1.01)
plt.tight_layout()
save_fig(fig, "11_numeric_histograms.png")
plt.show(); plt.close(fig)

print("\n[강의자료 Think to look for — 수치형 분포]")
print("  ✓ 이상점은 없는가?")
print("  ✓ 분포 모양 — 종모양? 치우침? 봉우리 수?")
print("  ✓ 히스토그램이 거칠다면 bin 수 조정 후 재확인")

---
## Part 12. 수치형 이상치 — 박스플롯 (IQR)
> 강의자료 p.39 IQR Based Outlier Detection

In [ ]:
n_c = 3; n_r = math.ceil(len(numeric_cols)/n_c)
fig, axes = plt.subplots(n_r, n_c, figsize=(18, n_r*4))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    ax = axes[i]
    sns.boxplot(x=cleaned[col], ax=ax, color="lightcoral",
                flierprops=dict(markersize=2, alpha=0.3))
    q1, q3 = cleaned[col].quantile(0.25), cleaned[col].quantile(0.75)
    iqr = q3 - q1
    n_out = ((cleaned[col]<q1-1.5*iqr)|(cleaned[col]>q3+1.5*iqr)).sum()
    ax.set_title(f"{col}\nIQR 이상치: {n_out:,}건 ({n_out/len(cleaned)*100:.1f}%)", fontsize=10)
    ax.set_xlabel("")

for j in range(i+1, len(axes)): fig.delaxes(axes[j])
fig.suptitle("수치형 컬럼 박스플롯 — IQR 이상치 탐지", fontsize=14, y=1.01)
plt.tight_layout()
save_fig(fig, "12_numeric_boxplots.png")
plt.show(); plt.close(fig)

print("\n[강의자료 Think to look for — 이상치]")
print("  ✓ 도메인에 따라 이상치가 중요 의미일 수 있음 (공항 고정요금 등)")
print("  ✓ 입력 오류/에러값이 Outlier로 인식될 수 있음")
print("  ✓ 무조건적 제거 위험 → 발생 원인 고찰 필요")

---
## Part 13. 수치형 vs 수치형 — 상관관계 히트맵
> 강의자료 p.25 변수 간 상관관계 탐색

In [ ]:
corr = cleaned[numeric_cols].corr()
mask = np.zeros_like(corr, dtype=bool)
mask[np.triu_indices_from(mask, k=1)] = True

fig, ax = plt.subplots(figsize=(13, 10))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, vmin=-1, vmax=1, mask=mask, ax=ax,
            linewidths=0.5, square=True)
ax.set_title("수치형 컬럼 간 상관관계 히트맵", fontsize=14)
plt.tight_layout()
save_fig(fig, "13_correlation_heatmap.png")
plt.show(); plt.close(fig)

print("\n[상관계수 |r|>0.7 쌍 — 다중공선성 후보]")
high_corr = (
    corr.where(np.tril(np.ones(corr.shape), k=-1).astype(bool))
    .stack().reset_index()
)
high_corr.columns = ["변수1","변수2","상관계수"]
display(high_corr[high_corr["상관계수"].abs()>0.7]
        .sort_values("상관계수", ascending=False))

print("\n[강의자료 Think to look for — 상관관계]")
print("  ✓ 가장 강한 상관관계를 가지는 변수는?")
print("  ✓ 서로 비슷한 정보 가진 변수 → 변수 제거, 다중공선성 검토")

---
## Part 14. 수치형 vs 수치형 — 산점도
> 강의자료 p.28 Quantitative vs Quantitative 살펴보기

In [ ]:
scatter_df = cleaned[
    (cleaned["trip_distance"]>0)&(cleaned["trip_distance"]<50)&
    (cleaned["fare_amount"]>0)&(cleaned["fare_amount"]<200)
].copy()
scatter_df["rate_label"] = scatter_df["RatecodeID"].map(LABEL_MAP["RatecodeID"]).fillna("기타")
sample_df = scatter_df.sample(min(30_000, len(scatter_df)), random_state=42)

fig = px.scatter(
    sample_df, x="trip_distance", y="fare_amount",
    color="rate_label",
    title="trip_distance vs fare_amount (요금 유형별 색 구분, 최대 3만건)",
    labels={"trip_distance":"이동 거리(마일)","fare_amount":"기본 요금($)","rate_label":"요금 유형"},
    opacity=0.4, color_discrete_sequence=px.colors.qualitative.Set1,
)
save_fig(fig, "14_scatter_distance_fare.png")
fig.show()

print("\n[강의자료 Think to look for — 산점도]")
print("  ✓ 선형 vs 비선형 관계?")
print("  ✓ 관계 강도는 강한가 약한가?")
print("  ✓ 이상점이 있는가? 로그변환 필요한가?")
print("  ✓ 인과관계 존재? 원인=X, 결과=Y")
print("  ✓ 데이터 너무 많으면 Sampling + alpha 옵션으로 반투명")
print("  → JFK(RatecodeID=2): fare≈70 고정, 거리와 무관")

---
## Part 15. 수치형 vs 범주형 — 그룹별 박스플롯
> 강의자료 p.29 Quantitative vs Categorical

In [ ]:
pairs = [
    ("payment_type", "fare_amount",   "결제 방식","기본 요금($)"),
    ("payment_type", "tip_amount",    "결제 방식","팁 금액($)"),
    ("VendorID",     "total_amount",  "벤더",    "총 요금($)"),
    ("RatecodeID",   "trip_distance", "요금 유형","이동 거리(마일)"),
]
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for i, (cat_col, num_col, xl, yl) in enumerate(pairs):
    ax = axes[i]
    plot_df = cleaned[[cat_col, num_col]].copy()
    if cat_col in LABEL_MAP:
        plot_df[cat_col] = plot_df[cat_col].map(LABEL_MAP[cat_col]).fillna(plot_df[cat_col].astype(str))
    p99 = plot_df[num_col].quantile(0.99)
    plot_df = plot_df[plot_df[num_col] <= p99]
    order = plot_df.groupby(cat_col)[num_col].median().sort_values().index

    sns.boxplot(data=plot_df, x=cat_col, y=num_col, order=order,
                ax=ax, palette="Set2", flierprops=dict(markersize=2, alpha=0.3))
    # 강의자료 권장: 점표시 + 반투명
    sns.stripplot(data=plot_df.sample(min(3000,len(plot_df)),random_state=42),
                  x=cat_col, y=num_col, order=order, ax=ax,
                  color="black", alpha=0.1, size=2)
    ax.set_title(f"{xl}별 {yl}\n(99th pct 이내 | 점=샘플3000)", fontsize=10)
    ax.set_xlabel(xl); ax.set_ylabel(yl)
    ax.tick_params(axis="x", rotation=20)

fig.suptitle("Quantitative vs Categorical — 그룹별 분포", fontsize=14, y=1.01)
plt.tight_layout()
save_fig(fig, "15_quant_vs_cat_boxplot.png")
plt.show(); plt.close(fig)

print("\n[강의자료 Think to look for — Quant vs Categorical]")
print("  ✓ Class별 이상치는 있는가?")
print("  ✓ Class별 관측치는 충분한가? (점표시+반투명 확인)")
print("  ✓ Class의 적절한 순서가 있는가?")
print("  ✓ 로그변환 / 축 교환 필요한가?")
print("  → 현금 결제: tip_amount≈0 → is_cash 파생변수 필요")

---
## Part 16. 핵심 인사이트 시각화
> Question-driven EDA (강의자료 p.31)

In [ ]:
# Q1. 시간대별 트립 수
hour_df = cleaned.copy()
hour_df["hour"] = hour_df["tpep_pickup_datetime"].dt.hour
hc = hour_df["hour"].value_counts().sort_index().reset_index()
hc.columns = ["hour","trip_count"]
fig = px.bar(hc, x="hour", y="trip_count",
             title="Q1. 시간대별 트립 수 — 러시아워 패턴",
             labels={"hour":"승차 시간대","trip_count":"트립 수"},
             color="trip_count", color_continuous_scale="Blues")
fig.update_layout(coloraxis_showscale=False)
save_fig(fig, "16a_trips_by_hour.png")
fig.show()
print("→ is_rush_hour 파생변수: 7~9시, 17~19시")

In [ ]:
# Q2. payment_type별 tip 분포
tip_df = cleaned.copy()
tip_df["payment_label"] = tip_df["payment_type"].map(LABEL_MAP["payment_type"]).fillna("기타")
fig = px.box(
    tip_df[tip_df["tip_amount"]>=0],
    x="payment_label", y="tip_amount",
    title="Q2. 결제 방식별 팁 금액 분포",
    labels={"payment_label":"결제 방식","tip_amount":"팁($)"},
    color="payment_label",
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.update_layout(showlegend=False, yaxis_range=[0, 20])
save_fig(fig, "16b_tip_by_payment.png")
fig.show()
print("→ 현금 결제: tip거의 0 → 팁 분석 시 신용카드(=1)만 사용")

In [ ]:
# Q3. 요일별 트립 수
dow_df = cleaned.copy()
dow_df["weekday"] = dow_df["tpep_pickup_datetime"].dt.dayofweek
dow_labels = {0:"월",1:"화",2:"수",3:"목",4:"금",5:"토",6:"일"}
dow_df["weekday_label"] = dow_df["weekday"].map(dow_labels)
dc = dow_df["weekday_label"].value_counts().reindex(["월","화","수","목","금","토","일"]).reset_index()
dc.columns = ["weekday","count"]
fig = px.bar(dc, x="weekday", y="count",
             title="Q3. 요일별 트립 수",
             labels={"weekday":"요일","count":"트립 수"},
             color="count", color_continuous_scale="Greens")
fig.update_layout(coloraxis_showscale=False)
save_fig(fig, "16c_trips_by_weekday.png")
fig.show()
print("→ weekday / is_weekend 파생변수 제안")

---
## Part 17. Borough 위치 분석

In [ ]:
borough_df = (
    mapped.groupby("PU_Borough",observed=True)
    .agg(trips=("total_amount","count"),avg_total=("total_amount","mean"))
    .reset_index().sort_values("trips",ascending=False)
)
fig = make_subplots(rows=1,cols=2,
                    subplot_titles=("Borough별 트립 수","Borough별 평균 total_amount($)"))
fig.add_trace(go.Bar(x=borough_df["PU_Borough"],y=borough_df["trips"],
                     marker_color="steelblue",name="트립 수"),row=1,col=1)
fig.add_trace(go.Bar(x=borough_df["PU_Borough"],y=borough_df["avg_total"].round(2),
                     marker_color="coral",name="평균 요금"),row=1,col=2)
fig.update_layout(title_text="Borough별 분석",showlegend=False,height=450)
save_fig(fig,"17_borough_analysis.png")
fig.show()

top_zones = mapped["PU_Zone"].value_counts().head(10).reset_index()
top_zones.columns = ["Zone","count"]
fig2 = px.bar(top_zones.sort_values("count"),x="count",y="Zone",orientation="h",
              title="픽업 트립 수 상위 10 Zone",
              labels={"count":"트립 수","Zone":"Zone"},
              color="count",color_continuous_scale="Blues")
fig2.update_layout(coloraxis_showscale=False)
save_fig(fig2,"17_top_zones.png")
fig2.show()

---
## Part 18. 관찰 정리 — 팀원 인계 메모

### 📌 EDA Findings (강의자료 p.13 기준)

| 질문 | 분석 | 발견 | Next Action |
|------|------|------|-------------|
| 결측치는 어디에? | isnull().sum() | 정제 완료 | — |
| 중복은? | duplicated() | 제거 완료 | — |
| 수치형 분포는? | 히스토그램+왜도 | trip_distance 등 오른쪽 치우침 | 로그변환 검토 |
| 이상치는? | IQR 박스플롯 | total_amount 음수, 소요시간 음수 | 이상치 처리 |
| 범주형 구성은? | value_counts | 신용카드 70%+ | 클래스 불균형 확인 |
| 비슷한 변수는? | 상관관계 히트맵 | fare ↔ total 강상관 | 다중공선성 검토 |

### 🛠 파생변수 제안 (강의자료 p.72 Feature Creation Strategies)

```python
# Transform Features — 날짜에서 시간 정보 추출
df['duration_min'] = (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']).dt.total_seconds()/60
df['hour']         = df['tpep_pickup_datetime'].dt.hour
df['weekday']      = df['tpep_pickup_datetime'].dt.dayofweek
df['is_weekend']   = df['weekday'].isin([5,6]).astype(int)

# Combine Features — 비율/차이
df['is_rush_hour'] = df['hour'].isin([7,8,9,17,18,19]).astype(int)
df['is_airport']   = df['RatecodeID'].isin([2,3]).astype(int)
df['is_cash']      = (df['payment_type']==2).astype(int)
df['tip_rate']     = df['tip_amount'] / df['fare_amount'].replace(0, float('nan'))
df['speed_mph']    = df['trip_distance'] / (df['duration_min']/60).replace(0, float('nan'))

# Transform Features — 구간화 (Binning)
df['fare_bin'] = pd.cut(df['fare_amount'], bins=[0,10,20,30,50,200],
                         labels=['~10','10~20','20~30','30~50','50+'])
```

### 📐 통계검정 제안

| 검정 | 가설 |
|------|------|
| t-test | 러시아워 vs 비러시아워 fare_amount 평균 차이 |
| ANOVA | Borough별 total_amount 평균 차이 |
| 카이제곱 | payment_type ↔ RatecodeID 독립성 |
| Pearson/Spearman | trip_distance ↔ fare_amount 상관계수 |
| 회귀분석 | total_amount 예측 모델 |